# Лабораторная работа № 4 — вариант 4

**Основной источник:** `https://www.scrapingcourse.com/javascript-rendering`  
**Механизм:** JavaScript-отрисовка каталога и загрузка данных через XHR/fetch.

На основной странице только 12 товаров. Поэтому требование **≥ 200 записей**
выполняется за счёт нескольких сценариев той же учебной площадки:

1. JavaScript Rendering — 12 товаров;
2. Pagination — все страницы каталога;
3. Infinite Scrolling — все товары после прокруток.

Одинаковый товар в разных сценариях считается отдельным наблюдением, потому что
проверяется разный контекст получения данных. Повторы по ключу товара отдельно
показываются в разделе контроля качества.

## 0. Импорты и настройки

Перед первым запуском глобального Python:

```powershell
python3.10 -m pip install playwright pandas requests beautifulsoup4 psutil
python3.10 -m playwright install chromium
```

In [1]:
import asyncio
import html as html_lib
import re
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin, urlsplit, urlunsplit

import pandas as pd
import psutil
import requests
from bs4 import BeautifulSoup
from IPython.display import display
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

BASE_URL = "https://www.scrapingcourse.com"
JS_URL = f"{BASE_URL}/javascript-rendering"
PAGINATION_URL = f"{BASE_URL}/pagination"
SCROLL_URL = f"{BASE_URL}/infinite-scrolling"
AJAX_JSON_URL = f"{BASE_URL}/ajax/products/json"
AJAX_HTML_URL = f"{BASE_URL}/ajax/products"

ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

USER_AGENT = "course-crawler/1.0 (educational use)"
HEADERS = {"User-Agent": USER_AGENT}
MAX_PAGES = 20
MAX_SCROLLS = 20
STALL_LIMIT = 2
MIN_RECORDS = 200

# В текущем окружении загружена полная сборка Chromium без headless-shell.
# На обычной установке Playwright этот параметр не понадобится.
LOCAL_CHROMIUM = (
    Path.home()
    / "AppData/Local/ms-playwright/chromium-1248/chrome-win64/chrome.exe"
)
LAUNCH_KWARGS = {"headless": True}
if LOCAL_CHROMIUM.exists():
    LAUNCH_KWARGS["executable_path"] = str(LOCAL_CHROMIUM)

print("Рабочая папка:", Path.cwd())
print("Chromium:", LAUNCH_KWARGS.get("executable_path", "стандартный Playwright"))

Рабочая папка: e:\Labs_2_course\Data_mining\lab_4
Chromium: C:\Users\k_dys\AppData\Local\ms-playwright\chromium-1248\chrome-win64\chrome.exe


## 1. Вспомогательные функции

In [2]:
def clean_text(value):
    """Убирает HTML-сущности и лишние пробелы."""
    if value is None:
        return None
    return " ".join(html_lib.unescape(str(value)).split())


def product_key(name):
    """Стабильный ключ товара для сверки разных способов сбора."""
    return re.sub(r"[^a-z0-9]+", "-", clean_text(name).lower()).strip("-")


def normalize_url(url):
    if not url:
        return None
    parts = urlsplit(urljoin(BASE_URL, url))
    host = parts.netloc.lower().removeprefix("www.")
    return urlunsplit(("https", host, parts.path.rstrip("/"), "", ""))


def parse_price(value):
    if value is None or value == "":
        return None
    if isinstance(value, (int, float)):
        return float(value)
    match = re.search(r"\d+(?:\.\d+)?", str(value).replace(",", "."))
    return float(match.group()) if match else None


def make_record(context, name, price, url=None, iteration=None):
    name = clean_text(name)
    key = product_key(name)
    return {
        "record_id": f"{context}:{key}",
        "context": context,
        "product_key": key,
        "name": name,
        "price": parse_price(price),
        "url": normalize_url(url),
        "iteration": iteration,
    }


def parse_cards_from_html(markup, context, iteration=None):
    soup = BeautifulSoup(markup, "html.parser")
    records = []
    for card in soup.select(".product-item"):
        name_el = card.select_one(".product-name, [data-testid='product-name']")
        price_el = card.select_one(".product-price, [data-testid='product-price']")
        link_el = card.select_one("a[href]")
        name = name_el.get_text(" ", strip=True) if name_el else ""
        if name:
            records.append(
                make_record(
                    context,
                    name,
                    price_el.get_text(" ", strip=True) if price_el else None,
                    link_el.get("href") if link_el else None,
                    iteration,
                )
            )
    return records


def deduplicate(records):
    return list({record["record_id"]: record for record in records}.values())


def process_tree_memory_mb():
    process = psutil.Process()
    processes = [process, *process.children(recursive=True)]
    total = 0
    for item in processes:
        try:
            total += item.memory_info().rss
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            continue
    return total / 1024**2


async def memory_monitor(samples, stop_event):
    while not stop_event.is_set():
        samples.append(process_tree_memory_mb())
        await asyncio.sleep(0.1)


async def save_timeout_artifacts(page, label):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = ARTIFACTS_DIR / f"{label}_{timestamp}"
    await page.screenshot(path=str(stem.with_suffix(".png")), full_page=True)
    stem.with_suffix(".html").write_text(await page.content(), encoding="utf-8")
    return str(stem)


async def extract_cards_from_page(page, context, iteration=None):
    raw = await page.locator(".product-item").evaluate_all(
        """cards => cards.map(card => ({
            name: card.querySelector('.product-name, [data-testid="product-name"]')?.textContent?.trim() ?? '',
            price: card.querySelector('.product-price, [data-testid="product-price"]')?.textContent?.trim() ?? '',
            url: card.querySelector('a[href]')?.href ?? ''
        }))"""
    )
    return [
        make_record(context, row["name"], row["price"], row["url"], iteration)
        for row in raw
        if clean_text(row["name"])
    ]


def attach_network_counter(page, stats):
    """Считает ответы и объём прочитанных тел ответов."""
    pending = set()

    async def read_response(response):
        stats["requests"] += 1
        try:
            stats["bytes"] += len(await response.body())
        except Exception as error:
            stats["unreadable_responses"] += 1
            stats["last_read_error"] = type(error).__name__

    def on_response(response):
        task = asyncio.create_task(read_response(response))
        pending.add(task)
        task.add_done_callback(pending.discard)

    page.on("response", on_response)
    return pending


async def wait_network_tasks(pending):
    if pending:
        await asyncio.gather(*list(pending))


def run_in_proactor(coroutine_factory):
    """Запускает Playwright в отдельном Windows ProactorEventLoop."""
    loop = asyncio.ProactorEventLoop()
    try:
        asyncio.set_event_loop(loop)
        return loop.run_until_complete(coroutine_factory())
    finally:
        loop.close()

## 2. Диагностика статического HTML

Сервер отдаёт 12 заготовок карточек, но их названия и цены пусты. В исходном
HTML также виден путь к JSON, который вызывается JavaScript-кодом.

In [3]:
session = requests.Session()
session.headers.update(HEADERS)

t0 = time.perf_counter()
server_response = session.get(JS_URL, timeout=30)
server_response.raise_for_status()
server_seconds = time.perf_counter() - t0
server_html = server_response.text
server_soup = BeautifulSoup(server_html, "html.parser")

server_cards = server_soup.select(".product-item")
server_names = [
    node.get_text(" ", strip=True)
    for node in server_soup.select("[data-testid='product-name']")
]

diagnostic_static = {
    "status": server_response.status_code,
    "html_bytes": len(server_response.content),
    "cards": len(server_cards),
    "nonempty_names": sum(bool(name) for name in server_names),
    "ajax_path_found": "/ajax/products/json" in server_html,
    "seconds": round(server_seconds, 3),
}
display(pd.DataFrame([diagnostic_static]))

assert diagnostic_static["cards"] == 12
assert diagnostic_static["nonempty_names"] == 0
assert diagnostic_static["ajax_path_found"]

,status,html_bytes,cards,nonempty_names,ajax_path_found,seconds
0,200,29656,12,0,True,12.072


## 3. Браузерный сбор Playwright

Используются три предохранителя: максимальное число страниц/прокруток,
остановка после двух итераций без прироста и ограничение по доступному каталогу.
Данные извлекаются одним вызовом JavaScript для всех карточек.

In [4]:
async def collect_browser(block_resources=True):
    records = []
    log = []
    stats = {"requests": 0, "bytes": 0, "unreadable_responses": 0}
    diagnostics = {}
    benchmark = {}
    memory_samples = []
    stop_memory = asyncio.Event()
    monitor_task = asyncio.create_task(memory_monitor(memory_samples, stop_memory))
    started = time.perf_counter()

    try:
        async with async_playwright() as playwright:
            browser = await playwright.chromium.launch(**LAUNCH_KWARGS)
            context = await browser.new_context(
                user_agent=USER_AGENT,
                viewport={"width": 1280, "height": 900},
                locale="ru-RU",
            )
            context.set_default_timeout(15_000)
            context.set_default_navigation_timeout(30_000)

            if block_resources:
                async def block_media(route):
                    if route.request.resource_type in {"image", "media", "font"}:
                        stats["blocked"] = stats.get("blocked", 0) + 1
                        await route.abort()
                    else:
                        await route.continue_()

                await context.route("**/*", block_media)

            # 3.1. Основная страница варианта 4.
            page = await context.new_page()
            pending = attach_network_counter(page, stats)
            try:
                async with page.expect_response(
                    lambda response: "/ajax/products/json" in response.url
                ) as response_info:
                    await page.goto(JS_URL, wait_until="domcontentloaded")
                ajax_response = await response_info.value
                await page.wait_for_function(
                    """() => [...document.querySelectorAll('[data-testid="product-name"]')]
                        .some(node => node.textContent.trim().length > 0)"""
                )

                js_records = await extract_cards_from_page(page, "javascript", 1)
                records.extend(js_records)
                diagnostics.update(
                    {
                        "browser_cards": len(js_records),
                        "xhr_url": ajax_response.url,
                        "xhr_method": ajax_response.request.method,
                        "xhr_status": ajax_response.status,
                        "xhr_content_type": ajax_response.headers.get("content-type"),
                    }
                )

                # Массовое и поэлементное извлечение на одних данных.
                bulk_start = time.perf_counter()
                await extract_cards_from_page(page, "javascript", 1)
                benchmark["bulk_seconds"] = time.perf_counter() - bulk_start

                per_item_start = time.perf_counter()
                cards = page.locator(".product-item")
                per_item_names = []
                for index in range(await cards.count()):
                    per_item_names.append(
                        await cards.nth(index).locator(".product-name").inner_text()
                    )
                benchmark["per_item_seconds"] = time.perf_counter() - per_item_start
                benchmark["items"] = len(per_item_names)

                log.append(
                    {
                        "method": "playwright",
                        "scenario": "javascript",
                        "iteration": 1,
                        "url": page.url,
                        "elements": len(js_records),
                        "unique_records": len(js_records),
                        "growth": len(js_records),
                        "duration_seconds": round(time.perf_counter() - started, 3),
                        "timestamp": datetime.now().isoformat(timespec="seconds"),
                        "status": "ok",
                    }
                )
            except PlaywrightTimeoutError:
                artifact = await save_timeout_artifacts(page, "javascript_timeout")
                log.append({"method": "playwright", "scenario": "javascript", "status": f"timeout: {artifact}"})
                raise

            # 3.2. Пагинация: переход по ссылке Next page.
            await page.goto(PAGINATION_URL, wait_until="domcontentloaded")
            pagination_seen = {}
            for page_number in range(1, MAX_PAGES + 1):
                iteration_started = time.perf_counter()
                await page.wait_for_selector(".product-item")
                batch = await extract_cards_from_page(page, "pagination", page_number)
                before = len(pagination_seen)
                pagination_seen.update({row["record_id"]: row for row in batch})
                growth = len(pagination_seen) - before
                log.append(
                    {
                        "method": "playwright",
                        "scenario": "pagination",
                        "iteration": page_number,
                        "url": page.url,
                        "elements": len(batch),
                        "unique_records": len(pagination_seen),
                        "growth": growth,
                        "duration_seconds": round(time.perf_counter() - iteration_started, 3),
                        "timestamp": datetime.now().isoformat(timespec="seconds"),
                        "status": "ok",
                    }
                )

                # Последняя страница содержит неполную порцию.
                if len(batch) < 12:
                    break
                next_link = page.get_by_role("link", name="Next page")
                if await next_link.count() == 0:
                    break
                old_url = page.url
                await next_link.click()
                await page.wait_for_url(lambda url: url != old_url)
                await asyncio.sleep(0.1)  # короткая пауза вежливости

            records.extend(pagination_seen.values())

            # 3.3. Бесконечная прокрутка.
            await page.goto(SCROLL_URL, wait_until="domcontentloaded")
            await page.wait_for_selector(".product-item")
            scroll_seen = {}
            stalls = 0

            initial = await extract_cards_from_page(page, "infinite_scroll", 0)
            scroll_seen.update({row["record_id"]: row for row in initial})

            for scroll_number in range(1, MAX_SCROLLS + 1):
                iteration_started = time.perf_counter()
                previous_elements = await page.locator(".product-item").count()
                before = len(scroll_seen)
                await page.locator("#end-of-page").scroll_into_view_if_needed()
                status = "ok"
                try:
                    await page.wait_for_function(
                        "previous => document.querySelectorAll('.product-item').length > previous",
                        arg=previous_elements,
                        timeout=5_000,
                    )
                except PlaywrightTimeoutError:
                    status = "no_growth"

                batch = await extract_cards_from_page(page, "infinite_scroll", scroll_number)
                scroll_seen.update({row["record_id"]: row for row in batch})
                growth = len(scroll_seen) - before
                current_elements = await page.locator(".product-item").count()

                log.append(
                    {
                        "method": "playwright",
                        "scenario": "infinite_scroll",
                        "iteration": scroll_number,
                        "url": page.url,
                        "elements": current_elements,
                        "unique_records": len(scroll_seen),
                        "growth": growth,
                        "duration_seconds": round(time.perf_counter() - iteration_started, 3),
                        "timestamp": datetime.now().isoformat(timespec="seconds"),
                        "status": status,
                    }
                )

                stalls = stalls + 1 if current_elements == previous_elements else 0
                if stalls >= STALL_LIMIT:
                    break
                await asyncio.sleep(0.1)  # короткая пауза вежливости

            records.extend(scroll_seen.values())
            await wait_network_tasks(pending)
            await context.close()
            await browser.close()
    finally:
        stop_memory.set()
        await monitor_task

    elapsed = time.perf_counter() - started
    stats.update(
        {
            "seconds": elapsed,
            "traffic_mb": stats["bytes"] / 1024**2,
            "peak_memory_mb": max(memory_samples, default=process_tree_memory_mb()),
        }
    )
    return deduplicate(records), log, stats, diagnostics, benchmark


# Jupyter на Windows использует SelectorEventLoop, который не запускает дочерние
# процессы. В рабочем потоке asyncio создаёт обычный ProactorEventLoop, поэтому
# Playwright остаётся асинхронным и корректно запускает браузер.
browser_records, browser_log, browser_stats, browser_diagnostics, extraction_benchmark = await asyncio.to_thread(
    run_in_proactor,
    lambda: collect_browser(block_resources=True),
)

df_browser = pd.DataFrame(browser_records).sort_values(["context", "product_key"]).reset_index(drop=True)
df_browser.to_csv("data_browser.csv", index=False, encoding="utf-8-sig")

print("Диагностика XHR:")
display(pd.DataFrame([browser_diagnostics]))
print("Число строк по сценариям:")
display(df_browser.groupby("context").size().rename("rows").to_frame())
print("Всего строк:", len(df_browser))

assert len(df_browser) >= MIN_RECORDS
assert len([row for row in browser_log if row.get("method") == "playwright"]) >= 5

Диагностика XHR:


,browser_cards,xhr_url,xhr_method,xhr_status,xhr_content_type
0,12,https://www.scrapingcourse.com/ajax/products/json,GET,200,application/json


Число строк по сценариям:


,rows
context,
infinite_scroll,147
javascript,12
pagination,147


Всего строк: 306


## 4. Проверка страницы без JavaScript

In [5]:
async def count_without_javascript():
    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(**LAUNCH_KWARGS)
        context = await browser.new_context(java_script_enabled=False, user_agent=USER_AGENT)
        page = await context.new_page()
        await page.goto(JS_URL, wait_until="domcontentloaded")
        cards = await page.locator(".product-item").count()
        nonempty = await page.locator("[data-testid='product-name']").evaluate_all(
            "nodes => nodes.filter(node => node.textContent.trim()).length"
        )
        await context.close()
        await browser.close()
        return {"cards": cards, "nonempty_names": nonempty}


javascript_disabled = await asyncio.to_thread(run_in_proactor, count_without_javascript)
display(pd.DataFrame([javascript_disabled], index=["JavaScript выключен"]))

,cards,nonempty_names
JavaScript выключен,12,0


## 5. Дешёвая альтернатива: JSON + HTTP

Браузер больше не нужен: основная страница берётся из найденного JSON,
пагинация — обычными GET-запросами, а бесконечный скролл — запросами к
`/ajax/products?offset=...`.

In [8]:
def collect_light():
    records = []
    log = []
    request_count = 0
    traffic_bytes = 0
    memory_samples = [process_tree_memory_mb()]
    started = time.perf_counter()

    http = requests.Session()
    http.headers.update(HEADERS)

    # JSON основной страницы.
    t0 = time.perf_counter()
    response = http.get(AJAX_JSON_URL, timeout=30)
    response.raise_for_status()
    request_count += 1
    traffic_bytes += len(response.content)
    json_rows = response.json()
    main_rows = [
        make_record("javascript", row["name"], row.get("sale_price") or row.get("regular_price"), iteration=1)
        for row in json_rows
    ]
    records.extend(main_rows)
    log.append(
        {
            "method": "requests",
            "scenario": "javascript_json",
            "iteration": 1,
            "url": response.url,
            "elements": len(main_rows),
            "unique_records": len(main_rows),
            "growth": len(main_rows),
            "duration_seconds": round(time.perf_counter() - t0, 3),
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "status": "ok",
        }
    )

    # Страницы пагинации.
    pagination_seen = {}
    for page_number in range(1, MAX_PAGES + 1):
        t0 = time.perf_counter()
        url = PAGINATION_URL if page_number == 1 else f"{PAGINATION_URL}/{page_number}"
        response = http.get(url, timeout=30)
        response.raise_for_status()
        request_count += 1
        traffic_bytes += len(response.content)
        batch = parse_cards_from_html(response.text, "pagination", page_number)
        before = len(pagination_seen)
        pagination_seen.update({row["record_id"]: row for row in batch})
        growth = len(pagination_seen) - before
        log.append(
            {
                "method": "requests",
                "scenario": "pagination",
                "iteration": page_number,
                "url": response.url,
                "elements": len(batch),
                "unique_records": len(pagination_seen),
                "growth": growth,
                "duration_seconds": round(time.perf_counter() - t0, 3),
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "status": "ok",
            }
        )
        memory_samples.append(process_tree_memory_mb())
        if len(batch) < 12:
            break
        time.sleep(0.1)
    records.extend(pagination_seen.values())

    # HTML-порции бесконечной прокрутки.
    scroll_seen = {}
    stalls = 0
    for iteration, offset in enumerate(range(0, 200, 10), start=1):
        t0 = time.perf_counter()
        response = http.get(AJAX_HTML_URL, params={"offset": offset}, timeout=30)
        if response.status_code == 204:
            batch = []
        else:
            response.raise_for_status()
            batch = parse_cards_from_html(response.text, "infinite_scroll", iteration)
        request_count += 1
        traffic_bytes += len(response.content)
        before = len(scroll_seen)
        scroll_seen.update({row["record_id"]: row for row in batch})
        growth = len(scroll_seen) - before
        log.append(
            {
                "method": "requests",
                "scenario": "infinite_scroll_ajax",
                "iteration": iteration,
                "url": response.url,
                "elements": len(batch),
                "unique_records": len(scroll_seen),
                "growth": growth,
                "duration_seconds": round(time.perf_counter() - t0, 3),
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "status": "ok" if batch else "empty_batch",
            }
        )
        memory_samples.append(process_tree_memory_mb())
        stalls = stalls + 1 if not batch else 0
        if stalls >= 1:
            break
        time.sleep(0.1)
    records.extend(scroll_seen.values())

    elapsed = time.perf_counter() - started
    stats = {
        "requests": request_count,
        "bytes": traffic_bytes,
        "traffic_mb": traffic_bytes / 1024**2,
        "seconds": elapsed,
        "peak_memory_mb": max(memory_samples),
    }
    return deduplicate(records), log, stats


light_records, light_log, light_stats = collect_light()
df_light = pd.DataFrame(light_records).sort_values(["context", "product_key"]).reset_index(drop=True)
df_light.to_csv("data_light.csv", index=False, encoding="utf-8-sig")

print("Число строк по сценариям:")
display(df_light.groupby("context").size().rename("rows").to_frame())
print("Всего строк:", len(df_light))

assert len(df_light) >= MIN_RECORDS

Число строк по сценариям:


,rows
context,
infinite_scroll,147
javascript,12
pagination,147


Всего строк: 306


## 6. Замер блокировки ресурсов

In [6]:
async def timed_dynamic_load(block_resources):
    stats = {"requests": 0, "bytes": 0, "unreadable_responses": 0, "blocked": 0}
    started = time.perf_counter()
    memory_samples = [process_tree_memory_mb()]

    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(**LAUNCH_KWARGS)
        context = await browser.new_context(user_agent=USER_AGENT)
        context.set_default_timeout(15_000)

        if block_resources:
            async def router(route):
                if route.request.resource_type in {"image", "media", "font"}:
                    stats["blocked"] += 1
                    await route.abort()
                else:
                    await route.continue_()
            await context.route("**/*", router)

        page = await context.new_page()
        pending = attach_network_counter(page, stats)
        await page.goto(JS_URL, wait_until="domcontentloaded")
        await page.wait_for_function(
            """() => [...document.querySelectorAll('[data-testid="product-name"]')]
                .some(node => node.textContent.trim().length > 0)"""
        )
        memory_samples.append(process_tree_memory_mb())
        await wait_network_tasks(pending)
        await context.close()
        await browser.close()

    stats["seconds"] = time.perf_counter() - started
    stats["traffic_mb"] = stats["bytes"] / 1024**2
    stats["peak_memory_mb"] = max(memory_samples)
    return stats


load_all = await asyncio.to_thread(run_in_proactor, lambda: timed_dynamic_load(False))
load_blocked = await asyncio.to_thread(run_in_proactor, lambda: timed_dynamic_load(True))
resource_comparison = pd.DataFrame(
    [
        {"mode": "all_resources", **load_all},
        {"mode": "blocked_media", **load_blocked},
    ]
)
display(resource_comparison)

,mode,requests,bytes,unreadable_responses,blocked,last_read_error,seconds,traffic_mb,peak_memory_mb
0,all_resources,13,882589,1,0,Error,3.265874,0.841702,1414.140625
1,blocked_media,11,875926,1,15,Error,2.514443,0.835348,1418.808594


## 7. Сверка наборов и проверка качества

In [9]:
browser_keys = set(df_browser["record_id"])
light_keys = set(df_light["record_id"])

comparison = {
    "browser_rows": len(df_browser),
    "light_rows": len(df_light),
    "only_browser": len(browser_keys - light_keys),
    "only_light": len(light_keys - browser_keys),
}

merged = df_browser.merge(
    df_light,
    on="record_id",
    how="inner",
    suffixes=("_browser", "_light"),
)
price_mismatches = merged[
    (merged["price_browser"].fillna(-1) - merged["price_light"].fillna(-1)).abs() > 0.001
]
comparison["price_mismatches"] = len(price_mismatches)
display(pd.DataFrame([comparison]))


def quality_report(frame):
    mandatory = ["context", "product_key", "name", "price"]
    empty_share = frame[mandatory].isna().mean().max()
    empty_share = max(empty_share, (frame["name"].astype(str).str.strip() == "").mean())
    return {
        "rows": len(frame),
        "unique_record_ids": frame["record_id"].nunique(),
        "unique_products": frame["product_key"].nunique(),
        "repeated_product_contexts": len(frame) - frame["product_key"].nunique(),
        "duplicate_record_ids": frame.duplicated("record_id").sum(),
        "max_empty_share": empty_share,
        "price_is_numeric": pd.api.types.is_numeric_dtype(frame["price"]),
    }


def smoke_check(frame, threshold=0.05):
    result = quality_report(frame)
    if result["max_empty_share"] > threshold:
        print(f"WARNING: доля пустых обязательных полей выше {threshold:.0%}")
    else:
        print(f"OK: доля пустых обязательных полей не выше {threshold:.0%}")
    return result


quality = pd.DataFrame(
    [
        {"method": "Playwright", **smoke_check(df_browser)},
        {"method": "HTTP", **smoke_check(df_light)},
    ]
)
display(quality)
assert comparison["only_browser"] == 0
assert comparison["only_light"] == 0
assert comparison["price_mismatches"] == 0

,browser_rows,light_rows,only_browser,only_light,price_mismatches
0,306,306,0,0,0


OK: доля пустых обязательных полей не выше 5%
OK: доля пустых обязательных полей не выше 5%


,method,rows,unique_record_ids,unique_products,repeated_product_contexts,duplicate_record_ids,max_empty_share,price_is_numeric
0,Playwright,306,306,147,159,0,0.0,True
1,HTTP,306,306,147,159,0,0.0,True


## 8. Итоговые метрики и журнал

In [10]:
collection_log = pd.DataFrame([*browser_log, *light_log])
collection_log.to_csv("collection_log.csv", index=False, encoding="utf-8-sig")

metrics = pd.DataFrame(
    [
        {
            "method": "Playwright",
            "records": len(df_browser),
            "seconds": browser_stats["seconds"],
            "requests": browser_stats["requests"],
            "traffic_mb": browser_stats["traffic_mb"],
            "peak_memory_mb": browser_stats["peak_memory_mb"],
        },
        {
            "method": "HTTP",
            "records": len(df_light),
            "seconds": light_stats["seconds"],
            "requests": light_stats["requests"],
            "traffic_mb": light_stats["traffic_mb"],
            "peak_memory_mb": light_stats["peak_memory_mb"],
        },
    ]
)
metrics["seconds_per_record"] = metrics["seconds"] / metrics["records"]
display(metrics.round(4))

benchmark_table = pd.DataFrame(
    [
        {
            "method": "bulk evaluate_all",
            "items": extraction_benchmark["items"],
            "seconds": extraction_benchmark["bulk_seconds"],
        },
        {
            "method": "per item locators",
            "items": extraction_benchmark["items"],
            "seconds": extraction_benchmark["per_item_seconds"],
        },
    ]
)
display(benchmark_table)

print("Созданы файлы:")
for filename in ["data_browser.csv", "data_light.csv", "collection_log.csv"]:
    path = Path(filename)
    print(f"- {path}: {path.stat().st_size} байт")

,method,records,seconds,requests,traffic_mb,peak_memory_mb,seconds_per_record
0,Playwright,306,31.5134,172,12.1838,1587.8594,0.1030
1,HTTP,306,12.2070,30,0.3525,173.4102,0.0399


,method,items,seconds
0,bulk evaluate_all,12,0.002516
1,per item locators,12,0.029732


Созданы файлы:
- data_browser.csv: 50427 байт
- data_light.csv: 49548 байт
- collection_log.csv: 7186 байт


## 9. Устойчивость локаторов

| Локатор | Почему выбран | Смена CSS-класса | Дополнительный `div` | Перевод текста |
|---|---|---|---|---|
| `[data-testid="product-name"]` | Специальный тестовый атрибут | Не влияет | Не влияет | Не влияет |
| `get_by_role("link", name="Next page")` | Семантика ссылки | Не влияет | Не влияет | Потребуется регулярное выражение или `rel="next"` |
| `#end-of-page` | Стабильный уникальный идентификатор sentinel | Не влияет, пока сохраняется `id` | Не влияет | Не влияет |

Для промышленного решения ссылку пагинации лучше искать по `rel="next"`, если
владелец сайта добавит этот атрибут. Структурный XPath и сгенерированные классы
не используются.

## 10. Вывод

`requests` не видит заполненные названия на основной странице, потому что они
появляются после запроса JavaScript к `/ajax/products/json`. Playwright нужен
для воспроизведения и проверки поведения страницы, но для регулярного сбора
дешевле обращаться к найденным JSON/HTML-эндпоинтам напрямую.